# Logistic regression with stable numerics

This notebook calls the functions in
[`mlshowcase/logistic.py`](../mlshowcase/logistic.py) and reads
[`results/logistic.json`](../results/logistic.json).
[`docs/logistic.md`](../docs/logistic.md) explains the experiment.

The protocol, rebuilt from Module 6 Assignment 6:

- The data is scikit-learn's bundled breast-cancer dataset, which needs no
  download. The source encodes malignant as 0 and benign as 1, and this project
  remaps malignant to positive class 1 and benign to 0, so precision, recall and
  F1 describe the malignant class.
- Seeds 42, 43, and 44 each produce a stratified split with 60% training,
  20% development, and 20% test data through `stratified_split`.
- The scaler fits training data only and applies the stored statistics to
  development and test data.
- Full-batch gradient descent minimizes mean binary cross-entropy at learning
  rates 0.01, 0.05, 0.1, and 0.5. Each fit runs for at most 3000 steps and
  stops earlier if its gradient meets the tolerance.
- Development positive-class F1 selects the learning rate and threshold together.
  The selected model then evaluates the test split once.
- An unregularized `LogisticRegression(C=np.inf)` baseline uses the same
  preprocessing and picks its own threshold on development.
- Do not use this experiment for clinical decisions.

The notebook never retrains the full experiment. It illustrates the reusable
API on small synthetic inputs and then reads the stored JSON and figures.


## Why the implementation is numerically stable

The historical notebook evaluated $\sigma(z) = 1 / (1 + e^{-z})$ directly,
which overflows `exp` for large negative scores. The reusable implementation
uses the algebraically identical two-branch form

$$
\sigma(z) =
\begin{cases}
1 / (1 + e^{-z}), & z \ge 0, \\
e^{z} / (1 + e^{z}), & z < 0,
\end{cases}
$$

so `exp` always receives a non-positive argument. The implementation does not
suppress warnings, and the formula cannot overflow.

The implementation computes cross-entropy from logits with `numpy.logaddexp`.
For a linear score $z_i = \theta_0 + x_i^\top \theta_{1:}$ and label
$y_i \in \{0, 1\}$,

$$
J(\theta) = \frac{1}{m} \sum_{i=1}^{m}
\begin{cases}
\mathrm{logaddexp}(0, -z_i), & y_i = 1, \\
\mathrm{logaddexp}(0, z_i), & y_i = 0,
\end{cases}
$$

which equals the usual cross-entropy but stays finite for large $|z_i|$. Its
gradient,

$$
\nabla_\theta J = \frac{1}{m} X^\top \left( \sigma(X\theta) - y \right),
$$

is `logistic_gradient`, and the objective is `logistic_loss`, so the two can be
compared with central finite differences.


In [ ]:
import json
import os
import sys
import tomllib
from pathlib import Path

# This notebook runs with its checkout's ``notebooks/`` directory as the working
# directory, so ``..`` is the public checkout root. Set MLSHOWCASE_ROOT to the
# checkout's absolute path to start the notebook from anywhere else.
EXPECTED_PROJECT = "mlshowcase"


def require_showcase_checkout(root: Path, *required_files: str) -> None:
    """Fail before any import or result read unless ``root`` is this checkout."""
    guide = (
        f"{root} is not the mlshowcase checkout this notebook belongs to: run the "
        "notebook with its notebooks/ directory as the working directory, or set "
        "MLSHOWCASE_ROOT to the checkout's absolute path"
    )
    pyproject = root / "pyproject.toml"
    if not pyproject.is_file():
        raise FileNotFoundError(f"{guide} (no pyproject.toml found there)")
    try:
        name = tomllib.loads(pyproject.read_text(encoding="utf-8"))["project"]["name"]
    except (KeyError, tomllib.TOMLDecodeError) as error:
        raise ValueError(f"{guide} (project.name could not be read)") from error
    if name != EXPECTED_PROJECT:
        raise ValueError(f"{guide} (project.name is {name!r})")
    for relative in required_files:
        if not (root / relative).is_file():
            raise FileNotFoundError(f"{guide} (missing {relative})")


_override = os.environ.get("MLSHOWCASE_ROOT")
if _override and not Path(_override).is_absolute():
    raise ValueError(f"MLSHOWCASE_ROOT must be an absolute path, not {_override!r}")
SHOWCASE_ROOT = Path(_override).resolve() if _override else Path("..").resolve()
require_showcase_checkout(SHOWCASE_ROOT, "mlshowcase/logistic.py", "notebooks/logistic.ipynb")

if str(SHOWCASE_ROOT) not in sys.path:
    sys.path.insert(0, str(SHOWCASE_ROOT))

import numpy as np  # noqa: E402

from mlshowcase.logistic import (  # noqa: E402
    FeatureScaler,
    add_intercept,
    cross_entropy,
    fit,
    logistic_gradient,
    logistic_loss,
    predict,
    predict_proba,
    sigmoid,
)

print("checkout root:", SHOWCASE_ROOT.name)


## Stability in practice

Extreme scores and logits are the cases that break the original `exp`-based
formula. Neither call below may raise an overflow warning, and both stay finite.


In [ ]:
extreme_scores = np.array([-1000.0, -40.0, 0.0, 40.0, 1000.0])
print("sigmoid:", sigmoid(extreme_scores))
print(
    "cross-entropy at extreme logits:",
    cross_entropy(np.array([0.0, 1.0]), np.array([1000.0, -1000.0])),
)

rng = np.random.default_rng(0)
design = add_intercept(rng.normal(size=(6, 2)))
labels = np.array([0.0, 1.0, 0.0, 1.0, 1.0, 0.0])
theta = np.array([0.2, -0.4, 0.6])
epsilon = 1e-6
numeric = np.array(
    [
        (
            logistic_loss(design, labels, theta + epsilon * step)
            - logistic_loss(design, labels, theta - epsilon * step)
        )
        / (2.0 * epsilon)
        for step in np.eye(theta.size)
    ]
)
print("analytic gradient:", logistic_gradient(design, labels, theta))
print("finite differences:", numeric)


## Train-only scaling

The historical `normalize` function computed separate means and standard
deviations for the train, development and test arrays, which leaked held-out
information into preprocessing. `FeatureScaler` fits once, stores `mean` and
`scale`, and transforms later data with those stored values. A constant column
has zero standard deviation, so its stored scale stays 1.0 and it transforms to
exact zeros instead of `NaN`.


In [ ]:
training = np.array([[1.0, 5.0], [1.0, 7.0], [1.0, 9.0]])
scaler = FeatureScaler.fit(training)
print("stored mean:", scaler.mean)
print("stored scale:", scaler.scale, "(constant column keeps scale 1.0)")
print("later rows scaled with the training statistics:")
print(scaler.transform(np.array([[1.0, 11.0], [1.0, -3.0]])))


## A tiny fit, for illustration only

`fit` runs handwritten full-batch gradient descent and returns a frozen
`FitResult` with the parameters and the optimization history. This synthetic
two-cloud problem shows the API; the experiment numbers in the next section
come from the generated JSON, not from this cell.


In [ ]:
rng = np.random.default_rng(4)
negative = rng.normal(loc=-1.5, scale=1.0, size=(40, 2))
positive = rng.normal(loc=1.5, scale=1.0, size=(40, 2))
demo_features = np.vstack((negative, positive))
demo_labels = np.concatenate((np.zeros(40), np.ones(40)))

result = fit(demo_features, demo_labels, learning_rate=0.5, max_steps=800, tolerance=1e-7)
print(
    f"converged={result.converged} steps={result.steps} "
    f"final loss={result.loss_history[-1]:.6f}"
)
probabilities = predict_proba(demo_features, result.theta)
print("first five probabilities:", np.round(probabilities[:5], 4))
for threshold in (0.1, 0.5, 0.9):
    positives = int(predict(demo_features, result.theta, threshold).sum())
    print(f"threshold {threshold}: {positives} positive predictions of 80")


## Generated evidence

The full experiment runs only through the guarded command line entry point:

```console
uv run --locked python -m mlshowcase.logistic --output-dir results
```

It writes `results/logistic.json` and the figures `results/logistic_loss.png`
and `results/logistic_comparison.png`. The JSON holds the protocol, per-seed
selections, held-out metrics, optimization curves and fit, selection and
evaluation timings. The cells below read those files and never retrain the
three-seed protocol; if the JSON is missing they raise `FileNotFoundError` with
the command that produces it, instead of falling back to example data. The
write-up lives in [`docs/logistic.md`](../docs/logistic.md).


In [ ]:
results_path = SHOWCASE_ROOT / "results" / "logistic.json"
if not results_path.is_file():
    raise FileNotFoundError(
        f"Missing {results_path.relative_to(SHOWCASE_ROOT)}. Run "
        "`uv run --locked python -m mlshowcase.logistic --output-dir results` from the "
        "repository root first."
    )

results = json.loads(results_path.read_text(encoding="utf-8"))
print("seeds:", results["protocol"]["splits"]["seeds"])
print("full class counts:", results["protocol"]["provenance"]["class_counts"])
print("label remap:", results["protocol"]["provenance"]["label_remap"])
print("environment:", results["environment"])


## Selections and held-out metrics

Selection uses the development split only, and the run scores the test split
once, after the learning rate and threshold are fixed. The handwritten model and
the baseline each choose their own threshold on development, so neither model is
scored against a threshold chosen for the other.


In [ ]:
for run in results["runs"]:
    selection = run["selection"]
    final = run["final"]
    baseline = run["baseline"]["final"]
    print(
        f"seed {run['seed']}: rate={selection['learning_rate']} "
        f"threshold={selection['threshold']} dev F1={selection['dev_f1']:.3f}"
    )
    print(
        "  custom   "
        f"accuracy={final['accuracy']:.3f} precision={final['positive_precision']:.3f} "
        f"recall={final['positive_recall']:.3f} F1={final['positive_f1']:.3f} "
        f"AUC={final['roc_auc']:.3f}"
    )
    print(
        "  baseline "
        f"accuracy={baseline['accuracy']:.3f} precision={baseline['positive_precision']:.3f} "
        f"recall={baseline['positive_recall']:.3f} F1={baseline['positive_f1']:.3f} "
        f"AUC={baseline['roc_auc']:.3f}"
    )


In [ ]:
summary = results["summary"]
for name in ("custom", "baseline"):
    print(
        name,
        {metric: round(values["mean"], 3) for metric, values in summary[name].items()},
    )
print("selection per seed:", summary["selections"])


In [ ]:
for run in results["runs"]:
    for rate in run["rates"]:
        print(
            f"seed {run['seed']} rate {rate['learning_rate']}: "
            f"steps={rate['steps']} converged={rate['converged']} "
            f"diverged={rate['diverged']} final loss={rate['final_loss']:.4f}"
        )
for run in results["runs"]:
    print(f"seed {run['seed']} timings (ms): {run['timings_ms']}")


## Figures

`logistic_loss.png` shows the training loss against the gradient-descent step
for every learning rate and seed: stable rates decrease smoothly, and a rate that
diverges shows up as a plateau or growth. `logistic_comparison.png` shows the
development F1 threshold curves with the selected points, and the held-out
metrics of the selected model next to the unregularized baseline.


In [ ]:
from IPython.display import Image, display

for figure_name in ("logistic_loss.png", "logistic_comparison.png"):
    figure_path = SHOWCASE_ROOT / "results" / figure_name
    if not figure_path.is_file():
        raise FileNotFoundError(
            f"Missing {figure_path.relative_to(SHOWCASE_ROOT)}; run the experiment "
            "command above first."
        )
    display(Image(filename=str(figure_path)))


## What the rebuild corrects

- The rebuild uses the stable two-branch form in place of the original
  `1 / (1 + np.exp(-z))`, and no code suppresses warnings.
- The original scaled train, development and test independently. `FeatureScaler`
  stores the training statistics once and applies them everywhere.
- The original printed evaluations from whatever `theta` was fitted last. This
  rebuild evaluates only the parameters of the selected learning rate, and only
  after the threshold is chosen.
- The original used scikit-learn's default, which made benign the positive
  class. The rebuild makes malignant positive class 1 and records that mapping
  in the protocol.
- The rebuild documents the gradient-norm tolerance, the step cap, and the
  divergence handling that excludes a non-finite run from selection.
- The rebuild repeats every configuration for three seeds and reports the
  repeat summary in the JSON.


## Limitations

- The bundled dataset is small and comes from a single source, so these numbers
  describe this protocol on this dataset only and make no clinical deployment
  claim.
- The model is unregularized linear logistic regression, so it cannot express
  interactions or nonlinear boundaries.
- Selection maximizes positive-class F1 on development, so a different
  criterion, for example recall at a fixed precision, would select different
  thresholds.
- Timings are wall-clock observations from one machine, not benchmarks.


## Reproducing

```console
uv run --locked python -m mlshowcase.logistic --output-dir results
uv run --locked pytest tests/test_logistic.py
```

The JSON records interpreter, operating-system, CPU, and package versions.
It stores no absolute paths, hostnames, or usernames.
